In [1]:
# =========================================
# CHILD SKIPPING TEST (DATA-DRIVEN)
# =========================================

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
from ultralytics import YOLO

In [2]:
# -------------------------------
# MediaPipe Setup
# -------------------------------
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils
DRAW_LM = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAW_CONN = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# -------------------------------
# Utils
# -------------------------------
def safe_mean(x): return float(np.mean(x)) if len(x) else 0.0
def safe_std(x): return float(np.std(x)) if len(x) else 0.0

def smooth(x, k=5):
    if len(x) < k: return np.array(x)
    return np.convolve(x, np.ones(k)/k, mode='same')

def angle(a,b,c):
    a,b,c = np.array(a),np.array(b),np.array(c)
    ba, bc = a-b, c-b
    cos = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-6)
    return np.degrees(np.arccos(np.clip(cos,-1,1)))

In [4]:
# Step-Hop Pattern
def step_score_cal(alt_ratio):
    step_score = 0
    if alt_ratio > 0.9: step_score = 5
    elif alt_ratio > 0.75: step_score = 4
    elif alt_ratio > 0.5: step_score = 3
    elif alt_ratio > 0.25: step_score = 2
    else: step_score = 1
    return step_score

In [5]:
# Rhythmic Flow
def rhythm_score_cal(rhythm_var):
    rhythm_score = 0
    if rhythm_var < 0.08: rhythm_score = 5
    elif rhythm_var < 0.12: rhythm_score = 4
    elif rhythm_var < 0.18: rhythm_score = 3
    elif rhythm_var < 0.25: rhythm_score = 2
    else: rhythm_score = 1
    return rhythm_score

In [6]:
# Arm Opposition
def arm_score_cal(corr):
    arm_score = 0
    if corr < -0.7: arm_score = 5
    elif corr < -0.5: arm_score = 4
    elif corr < -0.3: arm_score = 3
    elif corr < -0.1: arm_score = 2
    else: arm_score = 1
    return arm_score

In [7]:
# Fluid Movement
def fluid_score_cal(accel_var, jerk_var):
    fluid_score = 0
    if accel_var < 0.01 and jerk_var < 0.01: fluid_score = 5
    elif accel_var < 0.02: fluid_score = 4
    elif accel_var < 0.03: fluid_score = 3
    elif accel_var < 0.05: fluid_score = 2
    else: fluid_score = 1
    return fluid_score

In [8]:
def threshold_line(path):
    
    model = YOLO("best.pt")
    cap = cv2.VideoCapture(path)

    left_x, right_x = [], []
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.5, verbose=False)

        centers = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cx = (x1 + x2) // 2
            centers.append(cx)
            
            # Draw rectangle on cone
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Optional: confidence label
            cv2.putText(frame, f"Cone {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if len(centers) >= 2:
            centers.sort()
            left_x.append(centers[0])
            right_x.append(centers[-1])

        if len(left_x) > 50:
            break

        cv2.namedWindow("Video", cv2.WINDOW_NORMAL)
        cv2.setWindowProperty("Video", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
        cv2.imshow("Video", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Final fixed threshold
    left_avg = int(np.mean(left_x))
    right_avg = int(np.mean(right_x))
    # threshold_x = int((left_avg + right_avg) / 2)

    print("Left cone X:", left_avg)
    print("Right cone X:", right_avg)
    # print("Threshold X:", threshold_x)
    return left_avg, right_avg


In [12]:
# MAIN FUNCTION
def skipping_test(path="video.mp4"):

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    #threshold line calculate  
    left_line_x, right_line_x = 0.10, 0.85
    left_line_x, right_line_x = threshold_line(path)
    left_line_x = round((left_line_x/frame_width), 2)
    right_line_x = round((right_line_x/frame_width), 2) 
    print(f"{left_line_x} --- {right_line_x}")

    # signals
    hip_y = []
    l_ank_y, r_ank_y = [], []
    l_knee_y, r_knee_y = [], []
    l_el_y, r_el_y = [], []
    l_wri_y, r_wri_y = [], []

    frame_idx = 0
    window = "Skipping Analysis"
    cv2.namedWindow(window, cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)

        if res.pose_landmarks:
            lms = res.pose_landmarks.landmark

            # hip center
            hx = (lms[23].x + lms[24].x) / 2

            # boundary check
            if not (left_line_x <= hx <= right_line_x):
                continue

            # visibility filter
            if lms[23].visibility < 0.5:
                cv2.imshow(window, frame)
                if cv2.waitKey(1) & 0xFF == 27: break
                frame_idx += 1
                continue

            mpDraw.draw_landmarks(frame, res.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAW_LM, DRAW_CONN)

            # keypoints
            l_sh=(lms[11].x,lms[11].y); r_sh=(lms[12].x,lms[12].y)
            l_el=(lms[13].x,lms[13].y); r_el=(lms[14].x,lms[14].y)
            l_wri=(lms[15].x,lms[15].y); r_wri=(lms[16].x,lms[16].y)
            l_hip=(lms[23].x,lms[23].y); r_hip=(lms[24].x,lms[24].y)
            l_knee=(lms[25].x,lms[25].y); r_knee=(lms[26].x,lms[26].y)
            l_ank=(lms[27].x,lms[27].y); r_ank=(lms[28].x,lms[28].y)

            # COM
            hy = (l_hip[1] + r_hip[1]) / 2
            hip_y.append(hy)

            # legs
            l_ank_y.append(l_ank[1]); r_ank_y.append(r_ank[1])
            l_knee_y.append(l_knee[1]); r_knee_y.append(r_knee[1])

            # arms
            l_el_y.append(l_el[1]); r_el_y.append(r_el[1])
            l_wri_y.append(l_wri[1]); r_wri_y.append(r_wri[1])

        cv2.imshow(window, frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    # -------------------------------
    # PREPROCESS
    # -------------------------------
    hip_y = smooth(hip_y)
    l_ank_y = smooth(l_ank_y)
    r_ank_y = smooth(r_ank_y)

    n = min(len(hip_y), len(l_ank_y), len(r_ank_y))
    if n < 15:
        return 1,1,1,1

    hip_y = hip_y[:n]
    l_ank_y = l_ank_y[:n]
    r_ank_y = r_ank_y[:n]
    l_el_y = l_el_y[:n]
    r_el_y = r_el_y[:n]

    # -------------------------------
    # STEP-HOP SEGMENTATION
    # -------------------------------
    peaks,_ = find_peaks(-hip_y, distance=6)
    peaks = peaks[peaks < n]

    # determine lead foot sequence
    lead_seq = []
    for i in peaks:
        if l_ank_y[i] < r_ank_y[i]:
            lead_seq.append("L")
        else:
            lead_seq.append("R")

    # alternation quality
    alternations = sum(lead_seq[i] != lead_seq[i-1] for i in range(1,len(lead_seq)))
    alt_ratio = alternations / max(len(lead_seq)-1,1)

    # -------------------------------
    # RHYTHMIC FLOW (interval consistency)
    # -------------------------------
    if len(peaks) > 1:
        intervals = np.diff(peaks) / max(fps,1)
        rhythm_var = safe_std(intervals)
    else:
        rhythm_var = 1.0

    # -------------------------------
    # ARM-LEG OPPOSITION (correlation)
    # -------------------------------
    min_len = min(len(l_el_y), len(r_ank_y))
    corr = np.corrcoef(l_el_y[:min_len], r_ank_y[:min_len])[0,1]

    # -------------------------------
    # FLUID MOVEMENT (acceleration + jerk)
    # -------------------------------
    vel = np.diff(hip_y)
    accel = np.diff(vel)
    jerk = np.diff(accel) if len(accel) > 1 else np.array([0])

    accel_var = safe_std(accel)
    jerk_var = safe_std(jerk)

    # =========================================================
    # SCORING
    # =========================================================

    # Step-Hop Pattern
    step_score = step_score_cal(alt_ratio)

    # Rhythmic Flow
    rhythm_score = rhythm_score_cal(rhythm_var)

    # Arm Opposition
    arm_score = arm_score_cal(corr)

    # Fluid Movement
    fluid_score = fluid_score_cal(accel_var, jerk_var)

    print("------ SKIPPING RESULT ------")
    print("Step-Hop Pattern:", step_score)
    print("Rhythm:", rhythm_score)
    print("Arm Opposition:", arm_score)
    print("Fluid Movement:", fluid_score)

    final_score = (step_score + rhythm_score + arm_score + fluid_score)/4

    return final_score

In [14]:
path = "data/skipping.mp4"
print(skipping_test(path))

Left cone X: 72
Right cone X: 691
0.08 --- 0.81
------ SKIPPING RESULT ------
Step-Hop Pattern: 2
Rhythm: 3
Arm Opposition: 1
Fluid Movement: 4
2.5
